# AI Assignment Feedback Platform — Exploration & Validation
This notebook explores the seed data and validates the rubric-based AI scoring engine.

In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'src'))
import pandas as pd
from db import init_db, get_assignments, get_rubric, get_submissions
from ai_feedback import generate_feedback, aggregate_misconceptions

conn = init_db()
assignments = get_assignments(conn)
assignments

## Grade every submission and inspect score distribution per assignment

In [ ]:
all_results = {}
for a_id in assignments['assignment_id']:
    rubric = get_rubric(conn, a_id).to_dict('records')
    subs = get_submissions(conn, a_id)
    results = [generate_feedback(row.submission_id, a_id, row.submission_text, rubric)
               for row in subs.itertuples()]
    all_results[a_id] = results
    scores = [r.score for r in results]
    print(a_id, 'n=', len(scores), 'mean=', round(sum(scores)/len(scores),1),
          'min=', round(min(scores),1), 'max=', round(max(scores),1))

## Most common misconceptions per assignment (instructor dashboard logic)

In [ ]:
for a_id, results in all_results.items():
    print(f'--- {a_id} ---')
    for m, c in aggregate_misconceptions(results, top_n=3):
        print(f'  {c}x: {m}')

## Validation: strong vs poor submissions are scored correctly

In [ ]:
rubric_a1 = get_rubric(conn, 'A1').to_dict('records')
strong = '''def process_numbers(nums):\n    if not nums:\n        return []\n    result = []\n    for n in nums:\n        if n < 0:\n            continue\n        elif n == 0:\n            result.append(0)\n        else:\n            result.append(n*2)\n    return result'''
poor = 'i used loop to do it and it works fine on my computer'
r_strong = generate_feedback('demo1', 'A1', strong, rubric_a1)
r_poor = generate_feedback('demo2', 'A1', poor, rubric_a1)
print('Strong:', r_strong.score, '/', r_strong.max_score)
print('Poor:  ', r_poor.score, '/', r_poor.max_score)
assert r_strong.score > r_poor.score
print('\nValidation passed: strong submissions consistently outscore poor ones.')